# Leiden clustering (train split) for TNBC1 / TNBC2

This notebook:
1) Loads frozen clustering settings from `clearit/configs/leiden_clustering.yaml`
2) Loads train-split features from HDF5
3) Applies TNBC2-specific channel selection inline when dataset is TNBC2
4) Runs PCA, builds a kNN graph, and clusters with Leiden
5) Selects exemplars and saves all artifacts for downstream annotation and evaluation

In [1]:
from __future__ import annotations
from pathlib import Path
import json
import yaml
import numpy as np
import pandas as pd
import joblib

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR, REPO_ROOT
from clearit.leiden.io import list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph
from clearit.leiden.cluster import leiden
from clearit.leiden.exemplars import select_exemplars

# Reproducibility
SEED = 42

# Input config with frozen parameters
CONFIG_YAML = REPO_ROOT / "clearit" / "configs" / "leiden_clustering.yaml"

# HDF5 paths
H5_TNBC1 = EMBEDDINGS_DIR / "TNBC1-MxIF8"  / "inForm_MC7"    / "01_features-expressions" / "tnbc1-mxif8.hdf5"
H5_TNBC2 = EMBEDDINGS_DIR / "TNBC2-MIBI44" / "DeepCell_MC17" / "01_features-expressions" / "tnbc2-mibi8.hdf5"

# Output directories
BASE_OUT = OUTPUTS_DIR / "leiden"
BASE_OUT.mkdir(parents=True, exist_ok=True)

# Dataset selection: "TNBC1" or "TNBC2"
DATASET = "TNBC1"

# Train/test choice for this run (Phase 2 uses train only)
SPLIT = "train"

# Caps for large data; set to None to process all rows
PER_PATIENT_CAP = 10000
GLOBAL_CAP = 150000

print("Environment ready.")

Environment ready.


In [2]:
# Load frozen parameters
with open(CONFIG_YAML, "r") as f:
    cfg = yaml.safe_load(f)

assert DATASET in cfg, f"No settings found for dataset '{DATASET}' in {CONFIG_YAML}"
PCA_DIMS = int(cfg[DATASET]["PCA_DIMS"])
KNN_K    = int(cfg[DATASET]["KNN_K"])
LEI_RES  = float(cfg[DATASET]["LEI_RES"])

print(f"Using settings for {DATASET}: PCA={PCA_DIMS}, kNN-K={KNN_K}, res={LEI_RES}")

Using settings for TNBC1: PCA=64, kNN-K=70, res=0.6


In [3]:
# Resolve dataset path and patient groups
h5_path = H5_TNBC1 if DATASET.upper() == "TNBC1" else H5_TNBC2
assert h5_path.exists(), f"Missing file: {h5_path}"

patients_all = list_patients(h5_path)

# Patient split
if DATASET.upper() == "TNBC1" and len(patients_all) >= 62:
    train_pts = [p for p in patients_all if int(p[1:]) <= 47]
    test_pts  = [p for p in patients_all if p not in train_pts]
elif DATASET.upper() == "TNBC2" and len(patients_all) >= 41:
    train_pts = [p for p in patients_all if int(p[1:]) <= 30]
    test_pts  = [p for p in patients_all if p not in train_pts]
else:
    print("A")
    n_train = max(1, int(0.75 * len(patients_all)))
    train_pts, test_pts = patients_all[:n_train], patients_all[n_train:]

patients = train_pts if SPLIT == "train" else test_pts
print(f"{DATASET} | {SPLIT}: {len(patients)} patients")


TNBC1 | train: 47 patients


In [4]:
# Load features for the chosen split
X, meta, _, _ = load_hdf5_split(
    h5_path=h5_path,
    patients=patients,
    load_expressions=False,
    load_labels=False,
    per_patient_cap=PER_PATIENT_CAP,
    global_cap=GLOBAL_CAP,
    seed=SEED,
)
# TNBC2 channel selection
# Selects channels: Background, CD20, CD3, CD56, CD68, CD8, dsDNA, Pan-Keratin
if DATASET.upper() == "TNBC2":
    n_channels = 44
    n_features = 32
    channels_1_based = [1, 8, 10, 15, 17, 18, 19, 36]
    channels_0_based = [c - 1 for c in channels_1_based]

    assert X.shape[1] == n_channels * n_features, \
        f"Expected {n_channels*n_features} features, got {X.shape[1]}"
    Xr = X.reshape(X.shape[0], n_channels, n_features)
    Xsel = Xr[:, channels_0_based, :]
    X = Xsel.reshape(X.shape[0], -1)
print(f"Loaded features: X shape = {X.shape}, meta rows = {len(meta)}")
display(meta.head())

Loaded features: X shape = (150000, 256), meta rows = 150000


,patient,idx_within_patient
0,P01,10578
1,P01,23373
2,P01,26081
3,P01,17141
4,P01,8102


In [5]:
# PCA embedding
X_pca, scaler, pca = standardize_and_pca(X, n_components=PCA_DIMS, seed=SEED)
cum_var = float(pca.explained_variance_ratio_[:PCA_DIMS].sum())
print(f"PCA cumulative explained variance (k={PCA_DIMS}) = {cum_var:.4f}")

PCA cumulative explained variance (k=64) = 0.9575


In [6]:
# Build kNN graph and run Leiden
g = build_knn_graph(X_pca, k=KNN_K, metric="euclidean")
labels, n_clusters = leiden(g, resolution=LEI_RES, seed=SEED)

meta = meta.copy()
meta["cluster_id"] = labels
print(f"Leiden result: {n_clusters} clusters")

Leiden result: 14 clusters


In [7]:
# Cluster summary
cluster_sizes = meta["cluster_id"].value_counts().sort_index()
summary = (
    meta.groupby("cluster_id")["patient"]
    .nunique()
    .rename("n_patients")
    .to_frame()
    .assign(size=cluster_sizes.values)
    .reset_index()
    .sort_values("size", ascending=False)
)
print(f"Total clusters: {summary.shape[0]}")
display(summary.head(20))

Total clusters: 14


,cluster_id,n_patients,size
0,0,16,19640
1,1,15,17959
2,2,16,16908
3,3,16,14728
4,4,16,14126
5,5,14,12392
6,6,12,11537
7,7,16,11064
8,8,14,7414
9,9,13,7273


In [8]:
# Exemplar selection
EXEMPLARS_PER_CLUSTER = 10
EXEMPLARS_PER_PATIENT_MAX = 2
DENSITY_K = 15

exemplars = select_exemplars(
    X_pca=X_pca,
    meta=meta,
    labels=labels,
    exemplars_per_cluster=EXEMPLARS_PER_CLUSTER,
    per_patient_max=EXEMPLARS_PER_PATIENT_MAX,
    density_k=DENSITY_K,
)

print(
    f"Selected {len(exemplars)} exemplars across "
    f"{exemplars['cluster_id'].nunique()} clusters."
)
display(exemplars.head(20))


Selected 140 exemplars across 14 clusters.


,patient,idx_within_patient,cluster_id,exemplar_rank
40139,P05,24246,0,1.0
42699,P05,19721,0,2.0
58731,P06,3614,0,5.0
59993,P06,8776,0,10.0
103339,P11,3339,0,22.0
103352,P11,3352,0,25.0
149214,P16,18799,0,37.0
133100,P14,20236,0,63.0
146849,P16,11492,0,80.0
77257,P08,4366,0,102.0


In [11]:
# Save artifacts for downstream tasks
run_out = BASE_OUT / f"{DATASET.lower()}_{SPLIT}"
run_out.mkdir(parents=True, exist_ok=True)

# Tables
assign_csv    = run_out / "clusters_assignments.csv"
summary_csv   = run_out / "clusters_summary.csv"
exemplars_csv = run_out / "clusters_exemplars.csv"

meta.to_csv(assign_csv, index=False)
summary.to_csv(summary_csv, index=False)
exemplars.to_csv(exemplars_csv, index=False)

# Models and embeddings
pca_path     = run_out / "pca_model.joblib"
scaler_path  = run_out / "scaler.joblib"
X_pca_path   = run_out / "X_pca.npy"
centroids_np = run_out / "cluster_centroids.npy"

joblib.dump(pca, pca_path)
joblib.dump(scaler, scaler_path)
np.save(X_pca_path, X_pca)

# Centroids in PCA space for test-time assignment
centroids = (
    pd.DataFrame(X_pca)
    .assign(cluster_id=meta["cluster_id"].values)
    .groupby("cluster_id")
    .mean()
    .values
)
np.save(centroids_np, centroids)

# Settings log
settings_json = run_out / "settings.json"
with open(settings_json, "w") as f:
    json.dump(
        dict(
            dataset=DATASET,
            split=SPLIT,
            seed=SEED,
            pca_dims=PCA_DIMS,
            knn_k=KNN_K,
            leiden_resolution=LEI_RES,
            exemplars_per_cluster=EXEMPLARS_PER_CLUSTER,
            exemplars_per_patient_max=EXEMPLARS_PER_PATIENT_MAX,
            density_k=DENSITY_K,
            cumulative_pca_explained_variance=cum_var,
            n_clusters=int(n_clusters),
            n_rows=int(X.shape[0]),
            n_features=int(X.shape[1]),
        ),
        f,
        indent=2,
    )